# 18 — False Positive Error Analysis

Mục tiêu: phân tích các ảnh **Normal nhưng model dự đoán Pneumonia** trong external sanity test.

Notebook này:
- đọc `external_test_errors.csv`
- mở đúng các ảnh bị sai
- tính kích thước, brightness, contrast
- tạo contact sheet có filename + p(Pneumonia)
- lưu bảng thống kê để dễ tìm pattern chung

Không retrain. Không retune threshold. Chỉ phân tích lỗi.


In [ ]:
print('>>> CELL 1 START: config', flush=True)
from pathlib import Path
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image,ImageOps,ImageStat

ROOT=Path(r'D:\medicine')
ERROR_CSV=ROOT/'external_test_outputs'/'external_test_errors.csv'
OUT=ROOT/'external_test_outputs'/'fp_analysis'
OUT.mkdir(parents=True,exist_ok=True)

if not ERROR_CSV.exists():
    raise FileNotFoundError(f'Missing error CSV: {ERROR_CSV}')

errors=pd.read_csv(ERROR_CSV)
print('Errors:',len(errors),flush=True)
display(errors.head())


In [ ]:
print('>>> CELL 2 START: image statistics', flush=True)

rows=[]
for _,r in errors.iterrows():
    p=Path(r['path'])
    if not p.exists():
        raise FileNotFoundError(f'Missing image: {p}')

    with Image.open(p) as im:
        gray=im.convert('L')
        arr=np.asarray(gray,dtype=np.float32)
        rows.append({
            'filename':p.name,
            'path':str(p),
            'true_label':r['true_label'],
            'predicted_label':r['predicted_label'],
            'p_Pneumonia':float(r['p_Pneumonia']),
            'width':int(gray.width),
            'height':int(gray.height),
            'aspect_ratio':float(gray.width/max(gray.height,1)),
            'brightness_mean':float(arr.mean()),
            'contrast_std':float(arr.std()),
            'p01':float(np.percentile(arr,1)),
            'p99':float(np.percentile(arr,99)),
            'dynamic_range_1_99':float(np.percentile(arr,99)-np.percentile(arr,1)),
        })

stats=pd.DataFrame(rows).sort_values('p_Pneumonia',ascending=False).reset_index(drop=True)
stats.to_csv(OUT/'false_positive_image_stats.csv',index=False)
display(stats)


In [ ]:
print('>>> CELL 3 START: contact sheet', flush=True)

N=len(stats)
COLS=3
ROWS=math.ceil(N/COLS)

fig,axes=plt.subplots(ROWS,COLS,figsize=(15,5*ROWS))
axes=np.atleast_1d(axes).ravel()

for ax in axes:
    ax.axis('off')

for ax,(_,r) in zip(axes,stats.iterrows()):
    p=Path(r['path'])
    with Image.open(p) as im:
        gray=im.convert('L')
        ax.imshow(gray,cmap='gray')

    ax.set_title(
        f"{r['filename']}\n"
        f"p(Pneumonia)={r['p_Pneumonia']:.3f} | "
        f"{int(r['width'])}x{int(r['height'])}\n"
        f"brightness={r['brightness_mean']:.1f} | "
        f"contrast={r['contrast_std']:.1f}",
        fontsize=9
    )
    ax.axis('off')

plt.tight_layout()
sheet_path=OUT/'false_positive_contact_sheet.png'
plt.savefig(sheet_path,dpi=160,bbox_inches='tight')
plt.show()

print('Saved:',sheet_path,flush=True)


In [ ]:
print('>>> CELL 4 START: quick pattern summary', flush=True)

summary=pd.DataFrame({
    'metric':['p_Pneumonia','brightness_mean','contrast_std','width','height','aspect_ratio','dynamic_range_1_99'],
    'mean':[
        stats.p_Pneumonia.mean(),
        stats.brightness_mean.mean(),
        stats.contrast_std.mean(),
        stats.width.mean(),
        stats.height.mean(),
        stats.aspect_ratio.mean(),
        stats.dynamic_range_1_99.mean(),
    ],
    'min':[
        stats.p_Pneumonia.min(),
        stats.brightness_mean.min(),
        stats.contrast_std.min(),
        stats.width.min(),
        stats.height.min(),
        stats.aspect_ratio.min(),
        stats.dynamic_range_1_99.min(),
    ],
    'max':[
        stats.p_Pneumonia.max(),
        stats.brightness_mean.max(),
        stats.contrast_std.max(),
        stats.width.max(),
        stats.height.max(),
        stats.aspect_ratio.max(),
        stats.dynamic_range_1_99.max(),
    ],
})

summary.to_csv(OUT/'false_positive_summary_stats.csv',index=False)
display(summary)

print('\nHighest-confidence false positives:')
display(stats[['filename','p_Pneumonia','brightness_mean','contrast_std','width','height']].head(10))


## Cách đọc

Ưu tiên nhìn:
- ảnh nào có confidence rất cao nhưng vẫn là Normal
- nhóm ảnh có brightness/contrast khác biệt
- crop hoặc aspect ratio lạ
- viền đen, chữ, marker, thiết bị, tư thế chụp
- chất lượng ảnh hoặc framing khác phần lớn dataset

Nếu nhiều false positive có cùng đặc điểm, đó có thể là shortcut/domain feature mà model đã học.
